In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv("metric.csv", low_memory=False)
trace_df = pd.read_csv("trace.csv", low_memory=False)
log_df = pd.read_csv("log.csv", low_memory=False)
err_df = pd.read_csv("error_logs.csv", low_memory=False)

# Parse timestamps as UTC datetimes (Unix seconds -> datetime, UTC)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
err_df['timestamp'] = pd.to_datetime(err_df['timestamp'], unit='s', utc=True)

# --- metric.csv aggregates ---
metric_overview = pd.DataFrame({
    'total_rows': [len(metric_df)],
    'unique_cmdb_ids': [sorted(metric_df['cmdb_id'].unique())]
})

metric_top_kpis = metric_df['kpi_name'].value_counts().head(10).rename_axis('kpi_name').reset_index(name='count')
metric_examples = metric_df[['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(5)

# --- trace.csv aggregates ---
trace_overview = pd.DataFrame({
    'total_rows': [len(trace_df)],
    'unique_cmdb_ids': [sorted(trace_df['cmdb_id'].unique())]
})

trace_top_names = trace_df['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')
trace_examples = trace_df[['timestamp', 'cmdb_id', 'trace_name', 'value']].head(5)

# --- log.csv aggregates ---
log_overview = pd.DataFrame({
    'total_rows': [len(log_df)],
    'unique_cmdb_ids': [sorted(log_df['cmdb_id'].unique())]
})

log_top_names = log_df['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')
log_examples = log_df[['timestamp', 'cmdb_id', 'log_name', 'value']].head(5)

# --- error_logs.csv aggregates ---
err_overview = pd.DataFrame({
    'total_rows': [len(err_df)],
    'unique_cmdb_ids': [sorted(err_df['cmdb_id'].unique())]
})

err_examples = err_df[['timestamp', 'cmdb_id', 'message']].head(5)

# Return compact summaries (multiple variables so IPython displays them succinctly)
metric_overview, metric_top_kpis, metric_examples, trace_overview, trace_top_names, trace_examples, log_overview, log_top_names, log_examples, err_overview, err_examples

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: 1875 rows. CMDB IDs include carts, carts-db, catalogue, catalogue-db, front-end, orders, ...\n"
    "- Top metric KPIs by count: cpu (375), socket (375), mem (375), diskio (225), latency-50 (175), latency-90 (175), workload (175).\n"
    "- trace.csv: 0 rows (no trace telemetry available).\n"
    "- log.csv: 456 rows. CMDB IDs include carts, carts-db, catalogue, front-end, orders, ...; top log names: log.error_count (228) and log.row_count (228). Example log entries show zero error_count at the sampled timestamp.\n"
    "- error_logs.csv: 0 rows (no raw error log messages available).\n\n"
    "Implication: Metric and aggregated log features are available and should be the primary sources for root-cause investigation (focus on cpu, socket, mem, diskio and latency KPIs and services listed). Missing trace and error-log detail will limit deeper request-level debugging."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                                    unique_cmdb_ids
0        1875  [carts, carts-db, catalogue, catalogue-db, fro...,      kpi_name  count
0         cpu    375
1      socket    375
2         mem    375
3      diskio    225
4  latency-50    175
5  latency-90    175
6    workload    175,                   timestamp cmdb_id    kpi_name         value
0 2024-01-20 23:25:00+00:00   carts         cpu  1.663983e+00
1 2024-01-20 23:25:00+00:00   carts      diskio  0.000000e+00
2 2024-01-20 23:25:00+00:00   carts  latency-50  1.641907e-02
3 2024-01-20 23:25:00+00:00   carts  latency-90  3.413601e-02
4 2024-01-20 23:25:00+00:00   carts         mem  2.049420e+08,    total_rows unique_cmdb_ids
0           0              [], Empty DataFrame
Columns: [trace_name, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],    total_rows                                    unique_cmdb_ids
0         456  [carts, carts-db, catalogue, front-end, orders...,           log_name  count
0  log.error_count    228
1    log.row_count    228,                   timestamp    cmdb_id         log_name  value
0 2024-01-20 23:25:00+00:00      carts  log.error_count      0
1 2024-01-20 23:25:00+00:00      carts    log.row_count     25
2 2024-01-20 23:25:00+00:00   carts-db  log.error_count      0
3 2024-01-20 23:25:00+00:00   carts-db    log.row_count      1
4 2024-01-20 23:25:00+00:00  catalogue  log.error_count      0,    total_rows unique_cmdb_ids
0           0              [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing loaded DataFrames if present, otherwise load and parse timestamps
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv", low_memory=False)
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv", low_memory=False)
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 23:22:07", tz='UTC')
end = pd.Timestamp("2024-01-20 23:52:07", tz='UTC')

# --- 1) Metrics: compute global P95 and P5 per (cmdb_id, kpi_name) ---
metric_groups = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_global = metric_groups.quantile([0.95, 0.05]).unstack(level=-1)
# quantile result columns order might be [0.05,0.95] depending on index; normalize columns
metric_global = metric_global.rename(columns={0.95: 'global_p95', 0.05: 'global_p5'})
# In some pandas versions the columns might be floats; ensure both names exist
if 'global_p95' not in metric_global.columns or 'global_p5' not in metric_global.columns:
    # alternative extraction
    q = metric_groups.quantile([0.95, 0.05]).reset_index().pivot_table(index=['cmdb_id','kpi_name'], columns='level_2', values='value')
    metric_global = q.rename(columns={0.95: 'global_p95', 0.05: 'global_p5'})

metric_global = metric_global.reset_index()

# Filter metric rows to incident window
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Aggregate stats in window per group
m_w_grp = metric_window.groupby(['cmdb_id', 'kpi_name'])

m_window_stats = m_w_grp.agg(
    points_in_window=('value', 'count'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
).reset_index()

# Merge global thresholds
m_combined = pd.merge(metric_global, m_window_stats, on=['cmdb_id', 'kpi_name'], how='left')
# Fill NaN for groups not present in window with zeros/NaN for counts and min/max
m_combined['points_in_window'] = m_combined['points_in_window'].fillna(0).astype(int)
# For groups not in window, keep max/min as NaN

# Compute anomaly counts and earliest timestamps
# Prepare helper: join metric_window back to compute counts per group against thresholds
# Merge thresholds into window rows for easy boolean tests
metric_window_thresh = pd.merge(metric_window, metric_global, on=['cmdb_id', 'kpi_name'], how='left')

# High anomalies: value >= global_p95; Low anomalies: value <= global_p5
metric_window_thresh['is_high'] = metric_window_thresh['value'] >= metric_window_thresh['global_p95']
metric_window_thresh['is_low'] = metric_window_thresh['value'] <= metric_window_thresh['global_p5']

high_stats = metric_window_thresh[metric_window_thresh['is_high']].groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count_high=('is_high', 'sum'),
    earliest_high_timestamp=('timestamp', 'min')
).reset_index()

low_stats = metric_window_thresh[metric_window_thresh['is_low']].groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count_low=('is_low', 'sum'),
    earliest_low_timestamp=('timestamp', 'min')
).reset_index()

# Merge high/low stats into combined
m_combined = m_combined.merge(high_stats, on=['cmdb_id','kpi_name'], how='left')
m_combined = m_combined.merge(low_stats, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0
m_combined['anomaly_count_high'] = m_combined['anomaly_count_high'].fillna(0).astype(int)
m_combined['anomaly_count_low'] = m_combined['anomaly_count_low'].fillna(0).astype(int)

# Format earliest timestamps as ISO UTC strings or None
m_combined['earliest_high_timestamp'] = m_combined['earliest_high_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
m_combined['earliest_low_timestamp'] = m_combined['earliest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

m_combined['earliest_high_timestamp'] = m_combined['earliest_high_timestamp'].where(m_combined['anomaly_count_high']>0, None)
m_combined['earliest_low_timestamp'] = m_combined['earliest_low_timestamp'].where(m_combined['anomaly_count_low']>0, None)

# severity_high = max_value_in_window / global_p95 if global_p95>0 else None
def compute_sev(row):
    try:
        if pd.isna(row['global_p95']) or row['global_p95'] == 0:
            return None
        if pd.isna(row['max_value_in_window']):
            return None
        return row['max_value_in_window'] / row['global_p95']
    except Exception:
        return None

m_combined['severity_high'] = m_combined.apply(compute_sev, axis=1)

# Only return rows where anomaly_count_high>0 or anomaly_count_low>0
metric_anomalies = m_combined[(m_combined['anomaly_count_high']>0) | (m_combined['anomaly_count_low']>0)].copy()

# Select and order columns as requested
metric_anomalies = metric_anomalies[[
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p5', 'points_in_window',
    'anomaly_count_high', 'earliest_high_timestamp',
    'anomaly_count_low', 'earliest_low_timestamp',
    'max_value_in_window', 'min_value_in_window', 'severity_high'
]]

# Sort by severity_high desc then anomaly_count_high desc, limit top 20
metric_anom_df_top = metric_anomalies.sort_values(
    by=['severity_high', 'anomaly_count_high'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# --- 2) Logs: compute global P95 and P5 per (cmdb_id, log_name) ---
log_groups = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_global = log_groups.quantile([0.95, 0.05]).unstack(level=-1)
log_global = log_global.rename(columns={0.95: 'global_p95', 0.05: 'global_p5'})
if 'global_p95' not in log_global.columns or 'global_p5' not in log_global.columns:
    ql = log_groups.quantile([0.95, 0.05]).reset_index().pivot_table(index=['cmdb_id','log_name'], columns='level_2', values='value')
    log_global = ql.rename(columns={0.95: 'global_p95', 0.05: 'global_p5'})

log_global = log_global.reset_index()

# Filter log rows to incident window
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

lw_grp = log_window.groupby(['cmdb_id','log_name'])
log_window_stats = lw_grp.agg(
    points_in_window=('value','count'),
    max_value_in_window=('value','max'),
    min_value_in_window=('value','min')
).reset_index()

# Merge with global thresholds
log_combined = pd.merge(log_global, log_window_stats, on=['cmdb_id','log_name'], how='left')
log_combined['points_in_window'] = log_combined['points_in_window'].fillna(0).astype(int)

# Prepare threshold checks
log_window_thresh = pd.merge(log_window, log_global, on=['cmdb_id','log_name'], how='left')
log_window_thresh['is_high'] = log_window_thresh['value'] >= log_window_thresh['global_p95']
log_window_thresh['is_low'] = log_window_thresh['value'] <= log_window_thresh['global_p5']

log_high_stats = log_window_thresh[log_window_thresh['is_high']].groupby(['cmdb_id','log_name']).agg(
    anomaly_count_high=('is_high','sum'),
    earliest_high_timestamp=('timestamp','min')
).reset_index()

log_low_stats = log_window_thresh[log_window_thresh['is_low']].groupby(['cmdb_id','log_name']).agg(
    anomaly_count_low=('is_low','sum'),
    earliest_low_timestamp=('timestamp','min')
).reset_index()

log_combined = log_combined.merge(log_high_stats, on=['cmdb_id','log_name'], how='left')
log_combined = log_combined.merge(log_low_stats, on=['cmdb_id','log_name'], how='left')

log_combined['anomaly_count_high'] = log_combined['anomaly_count_high'].fillna(0).astype(int)
log_combined['anomaly_count_low'] = log_combined['anomaly_count_low'].fillna(0).astype(int)

log_combined['earliest_high_timestamp'] = log_combined['earliest_high_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_combined['earliest_low_timestamp'] = log_combined['earliest_low_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

log_combined['earliest_high_timestamp'] = log_combined['earliest_high_timestamp'].where(log_combined['anomaly_count_high']>0, None)
log_combined['earliest_low_timestamp'] = log_combined['earliest_low_timestamp'].where(log_combined['anomaly_count_low']>0, None)

# Select requested columns
log_anomalies = log_combined[[
    'cmdb_id', 'log_name', 'global_p95', 'global_p5', 'points_in_window',
    'anomaly_count_high', 'earliest_high_timestamp',
    'anomaly_count_low', 'earliest_low_timestamp',
    'max_value_in_window', 'min_value_in_window'
]]

# Only return rows where anomaly_count_high>0 or anomaly_count_low>0
log_anom_df_top = log_anomalies[(log_anomalies['anomaly_count_high']>0) | (log_anomalies['anomaly_count_low']>0)].copy()

# Sort by anomaly_count_high desc and limit top 20
log_anom_df_top = log_anom_df_top.sort_values(by=['anomaly_count_high'], ascending=False).head(20).reset_index(drop=True)

# --- 3) Tiny summary: cmdb_id with any metric anomalies and count of distinct anomalous KPIs ---
metric_anom_flags = metric_anomalies.copy()
metric_anom_flags['anomalous'] = True
comp_summary = metric_anom_flags.groupby('cmdb_id').agg(
    anomalous_kpi_count=('kpi_name','nunique')
).reset_index().sort_values(by='anomalous_kpi_count', ascending=False).head(10).reset_index(drop=True)

# Keep outputs compact
metric_anom_df_top, log_anom_df_top, comp_summary

```
Out[2]:
```
Summary (incident window 2024-01-20T23:22:07 – 2024-01-20T23:52:07 UTC):

Key metric anomalies (top candidates)
- user / diskio — highest severity_high (≈13.40). global_p95=6428.40, points_in_window=25, anomaly_count_high=2 (earliest high 2024-01-20T23:25:00Z), anomaly_count_low=23 (earliest low 2024-01-20T23:27:00Z). max_in_window=86120.12.
- queue-master / diskio — severity_high ≈1.55. global_p95=75947.29, anomaly_count_high=2 (earliest 2024-01-20T23:40:00Z). max_in_window=117999.0.
- rabbitmq-exporter / cpu — severity_high ≈1.25, 2 high anomalies (earliest 2024-01-20T23:26:00Z).
- queue-master / cpu, carts / cpu, orders / cpu, shipping / cpu, user / cpu, catalogue / cpu, catalogue-db / cpu — all show 2 high anomalies in-window with modest severity (≈1.02–1.21).
- Additional metric anomalies include socket, workload and diskio on several services (see results for full top-20 list).

Notes about metrics: most anomaly groups had 25 points in-window (full minute series). Several components show small numbers of high spikes (anomaly_count_high = 2) but some also have many low values relative to global P5.

Log-level findings
- Many services report log.error_count with global_p95 = global_p5 = 0.0. Because thresholds are 0, every in-window value equals threshold and the code flags both "high" and "low" (anomaly_count_high and anomaly_count_low = points_in_window). These entries are not informative by themselves (max and min are 0).
- log.row_count shows meaningful deviations:
  - carts-db / log.row_count: points_in_window=24, anomaly_count_high=24 (earliest 2024-01-20T23:25:00Z), max=2, min=1.
  - front-end / log.row_count: global_p95≈2026.4, points_in_window=25, anomaly_count_high=2 (earliest 2024-01-20T23:26:00Z), max=2077 (indicates short high spikes).
  - queue-master, orders, shipping, payment also show log.row_count high anomalies (several spikes during window).

Components with most distinct anomalous KPIs (top 10)
- carts: 7 distinct anomalous KPIs
- user: 7
- catalogue: 6
- orders: 6
- shipping: 6
- payment: 6
- front-end: 6
- orders-db: 4
- catalogue-db: 4
- carts-db: 4

Suggested next steps
- Prioritize investigation of user disk I/O (largest severity) and queue-master disk I/O — inspect I/O subsystem and recent deployments/config affecting those hosts.
- Examine CPU spikes across multiple services (queue-master, carts, orders, shipping, rabbitmq, etc.) — correlate with traffic/workload changes.
- Treat log.error_count results cautiously because thresholds are zero; instead focus on log.row_count spikes for front-end, queue-master, orders, shipping, payment for request-rate or processing changes.
- Because trace and raw error_logs were empty, request-level traces / error messages are not available; use metrics + logs above to guide targeted debugging (I/O, CPU, workload) on the listed components.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  kpi_name    global_p95     global_p5  points_in_window  anomaly_count_high earliest_high_timestamp  anomaly_count_low earliest_low_timestamp  max_value_in_window  min_value_in_window  severity_high
0                user    diskio  6.428398e+03  0.000000e+00                25                   2    2024-01-20T23:25:00Z                 23   2024-01-20T23:27:00Z         8.612012e+04         0.000000e+00      13.396824
1        queue-master    diskio  7.594729e+04  2.634368e+04                25                   2    2024-01-20T23:40:00Z                  2   2024-01-20T23:47:00Z         1.179990e+05         2.000232e+04       1.553696
2   rabbitmq-exporter       cpu  4.802123e-03  1.296311e-04                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:31:00Z         6.009916e-03         1.154986e-04       1.251512
3        queue-master       cpu  5.681389e-01  3.060416e-01                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:47:00Z         6.888182e-01         2.727517e-01       1.212412
4               carts       cpu  1.782347e+00  1.391448e+00                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:38:00Z         2.034707e+00         1.376912e+00       1.141589
5           orders-db    diskio  8.765148e+05  7.406069e+05                25                   2    2024-01-20T23:25:00Z                  2   2024-01-20T23:28:00Z         9.613699e+05         6.937963e+05       1.096810
6              orders       cpu  1.701799e+00  1.032595e+00                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:41:00Z         1.862152e+00         1.021623e+00       1.094226
7            shipping       cpu  7.207981e-01  3.303841e-01                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:41:00Z         7.830888e-01         3.235918e-01       1.086419
8              orders    socket  2.064333e+01  1.444667e+01                25                   2    2024-01-20T23:39:00Z                  2   2024-01-20T23:28:00Z         2.231667e+01         1.408333e+01       1.081059
9                user       cpu  9.461599e-01  7.889537e-01                25                   2    2024-01-20T23:25:00Z                  2   2024-01-20T23:32:00Z         1.014527e+00         7.405868e-01       1.072258
10           carts-db    diskio  2.244511e+06  1.881715e+06                25                   2    2024-01-20T23:37:00Z                  2   2024-01-20T23:25:00Z         2.348841e+06         1.795868e+06       1.046482
11           rabbitmq       cpu  8.829991e-01  7.423052e-01                25                   2    2024-01-20T23:30:00Z                  2   2024-01-20T23:29:00Z         9.219107e-01         7.251175e-01       1.044068
12           shipping    socket  6.586667e+00  5.030000e+00                25                   2    2024-01-20T23:33:00Z                  2   2024-01-20T23:28:00Z         6.866667e+00         5.000000e+00       1.042510
13           shipping  workload  2.130670e+00  1.812347e+00                25                   2    2024-01-20T23:29:00Z                  2   2024-01-20T23:44:00Z         2.217700e+00         1.779917e+00       1.040846
14           carts-db       cpu  4.278756e+00  3.758474e+00                25                   2    2024-01-20T23:43:00Z                  2   2024-01-20T23:29:00Z         4.450900e+00         3.658864e+00       1.040232
15          orders-db       cpu  4.614128e-01  3.970822e-01                25                   2    2024-01-20T23:36:00Z                  2   2024-01-20T23:28:00Z         4.770400e-01         3.856453e-01       1.033868
16               user  workload  1.902583e+01  1.621711e+01                25                   2    2024-01-20T23:27:00Z                  2   2024-01-20T23:46:00Z         1.962432e+01         1.588427e+01       1.031457
17          catalogue       cpu  1.943798e-01  1.695698e-01                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:41:00Z         2.001803e-01         1.645810e-01       1.029841
18             orders  workload  2.123500e+00  1.808807e+00                25                   2    2024-01-20T23:27:00Z                  2   2024-01-20T23:46:00Z         2.185750e+00         1.798900e+00       1.029315
19       catalogue-db       cpu  2.272636e-01  2.013653e-01                25                   2    2024-01-20T23:28:00Z                  2   2024-01-20T23:46:00Z         2.320944e-01         2.003891e-01       1.021256,          cmdb_id         log_name  global_p95  global_p5  points_in_window  anomaly_count_high earliest_high_timestamp  anomaly_count_low earliest_low_timestamp  max_value_in_window  min_value_in_window
0      front-end  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
1      catalogue  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
2        payment  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
3           user  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
4       shipping  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
5   queue-master  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
6         orders  log.error_count         0.0        0.0                25                  25    2024-01-20T23:25:00Z                 25   2024-01-20T23:25:00Z                    0                    0
7       carts-db    log.row_count         1.0        1.0                24                  24    2024-01-20T23:25:00Z                 23   2024-01-20T23:25:00Z                    2                    1
8       carts-db  log.error_count         0.0        0.0                24                  24    2024-01-20T23:25:00Z                 24   2024-01-20T23:25:00Z                    0                    0
9          carts  log.error_count         0.0        0.0                21                  21    2024-01-20T23:25:00Z                 21   2024-01-20T23:25:00Z                    0                    0
10    session-db  log.error_count         0.0        0.0                 5                   5    2024-01-20T23:27:00Z                  5   2024-01-20T23:27:00Z                    0                    0
11    session-db    log.row_count         5.0        5.0                 5                   5    2024-01-20T23:27:00Z                  5   2024-01-20T23:27:00Z                    5                    5
12      shipping    log.row_count        64.0       51.2                25                   4    2024-01-20T23:28:00Z                  2   2024-01-20T23:37:00Z                   65                    4
13       payment    log.row_count       104.0       89.4                25                   4    2024-01-20T23:28:00Z                  2   2024-01-20T23:25:00Z                  105                   10
14  queue-master    log.row_count       576.0      460.8                25                   4    2024-01-20T23:28:00Z                  2   2024-01-20T23:37:00Z                  585                   36
15        orders    log.row_count       128.0      102.4                25                   4    2024-01-20T23:28:00Z                  2   2024-01-20T23:37:00Z                  130                    8
16     front-end    log.row_count      2026.4     1725.8                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:37:00Z                 2077                  228
17     catalogue    log.row_count       165.8      144.0                25                   2    2024-01-20T23:26:00Z                  3   2024-01-20T23:25:00Z                  169                   16
18         carts    log.row_count       150.0       25.0                21                   2    2024-01-20T23:41:00Z                  6   2024-01-20T23:25:00Z                  150                   25
19          user    log.row_count       614.0      522.8                25                   2    2024-01-20T23:26:00Z                  2   2024-01-20T23:37:00Z                  618                   63,         cmdb_id  anomalous_kpi_count
0         carts                    7
1          user                    7
2     catalogue                    6
3        orders                    6
4      shipping                    6
5       payment                    6
6     front-end                    6
7     orders-db                    4
8  catalogue-db                    4
9      carts-db                    4)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load and parse
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv", low_memory=False)
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp dtype is correct
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 23:22:07", tz='UTC')
end = pd.Timestamp("2024-01-20 23:52:07", tz='UTC')

# Candidate list
candidates = [
    ("user","diskio"),
    ("queue-master","diskio"),
    ("rabbitmq-exporter","cpu"),
    ("carts","cpu"),
    ("orders","cpu"),
    ("user","cpu")
]

rows = []
summary_rows = []

for cmdb_id, kpi_name in candidates:
    # Compute global thresholds from full series (requirement)
    sel_full = metric_df[(metric_df['cmdb_id'] == cmdb_id) & (metric_df['kpi_name'] == kpi_name)]
    if sel_full.empty:
        global_p95 = np.nan
        global_p5 = np.nan
    else:
        global_p95 = sel_full['value'].quantile(0.95)
        global_p5 = sel_full['value'].quantile(0.05)
    # Filter to incident window
    sel_win = sel_full[(sel_full['timestamp'] >= start) & (sel_full['timestamp'] <= end)].copy()
    # Sort by timestamp
    sel_win = sel_win.sort_values('timestamp').reset_index(drop=True)
    # Compute is_high / is_low (comparison with thresholds computed on full series)
    sel_win['global_p95'] = global_p95
    sel_win['global_p5'] = global_p5
    sel_win['is_high'] = sel_win['value'] >= sel_win['global_p95']
    sel_win['is_low'] = sel_win['value'] <= sel_win['global_p5']
    # Format timestamp to ISO UTC
    sel_win['timestamp'] = sel_win['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Keep requested columns
    sel_out = sel_win[['cmdb_id','kpi_name','global_p95','global_p5','timestamp','value','is_high','is_low']].copy()
    # Append rows for combined table
    if not sel_out.empty:
        rows.append(sel_out)
    else:
        # create empty frame with columns to keep consistent output (no in-window rows)
        rows.append(pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','global_p5','timestamp','value','is_high','is_low']))
    # Compute earliest_high_timestamp and count of consecutive high sequences (runs)
    # For run count we need boolean series in time order
    is_high_series = sel_win['is_high'].astype(bool) if not sel_win.empty else pd.Series(dtype=bool)
    if is_high_series.empty:
        earliest_high = None
        high_runs = 0
    else:
        if is_high_series.any():
            # earliest high timestamp (from sel_win before formatting)
            # need original timestamps from sel_full filtered window
            orig_ts = sel_full[(sel_full['timestamp'] >= start) & (sel_full['timestamp'] <= end)].sort_values('timestamp')['timestamp']
            earliest_high_ts = orig_ts[is_high_series.values].min() if is_high_series.any() else None
            earliest_high = earliest_high_ts.strftime('%Y-%m-%dT%H:%M:%SZ') if pd.notna(earliest_high_ts) else None
            # count runs: a run starts where current is True and previous is False (or it's the first element)
            prev = is_high_series.shift(fill_value=False)
            runs = ((~prev) & is_high_series).sum()
            high_runs = int(runs)
        else:
            earliest_high = None
            high_runs = 0

    summary_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'global_p95': global_p95,
        'global_p5': global_p5,
        'earliest_high_timestamp': earliest_high,
        'high_run_count': high_runs,
        'points_in_window': len(sel_win)
    })

# Combine per-candidate row tables into one DataFrame with a separator grouping
combined_df = pd.concat(rows, keys=[f"{c[0]}|{c[1]}" for c in candidates], names=['candidate']).reset_index(level=0).drop(columns=['level_1'], errors='ignore').reset_index(drop=True)

# Summary DataFrame
summary_df = pd.DataFrame(summary_rows)[['cmdb_id','kpi_name','global_p95','global_p5','points_in_window','earliest_high_timestamp','high_run_count']]

# Display compact outputs: combined per-point rows (all in-window rows for candidates) and the per-candidate summary
combined_df, summary_df

```
Out[3]:
```
Summary for the requested candidates (incident window 2024-01-20T23:22:07 — 2024-01-20T23:52:07Z):

1) user / diskio
- global_p95 = 6428.398179, global_p5 = 0.0
- points in window = 25
- earliest high (value >= global_p95): 2024-01-20T23:25:00Z
- count of separate high runs = 1
- Notes: two high samples in-window (e.g., 2024-01-20T23:25:00Z value 86120.116709, 2024-01-20T23:26:00Z value 8035.497724). Many zeros in-window flagged low (<= global_p5).

2) queue-master / diskio
- global_p95 = 75947.292512, global_p5 = 26343.682367
- points in window = 25
- earliest high: 2024-01-20T23:40:00Z
- high run count = 1
- Notes: two high samples around 23:40–23:41; max in-window ~117999.

3) rabbitmq-exporter / cpu
- global_p95 = 0.004802123, global_p5 = 0.0001296311
- points in window = 25
- earliest high: 2024-01-20T23:26:00Z
- high run count = 2
- Notes: small absolute cpu numbers but multiple separate high runs (two distinct contiguous high sequences).

4) carts / cpu
- global_p95 = 1.782347, global_p5 = 1.391448
- points in window = 25
- earliest high: 2024-01-20T23:26:00Z
- high run count = 2

5) orders / cpu
- global_p95 = 1.701799, global_p5 = 1.032595
- points in window = 25
- earliest high: 2024-01-20T23:26:00Z
- high run count = 2

6) user / cpu
- global_p95 = 0.946160, global_p5 = 0.788954
- points in window = 25
- earliest high: 2024-01-20T23:25:00Z
- high run count = 2
- Note: last sampled value at 2024-01-20T23:49:00Z is high as well.

All per-point rows for these candidates (timestamp, value, is_high, is_low) were produced (≤25 rows per candidate). Key takeaway: user/diskio shows the largest spike (very high absolute disk I/O at 23:25Z), queue-master/diskio shows notable disk spikes later (~23:40Z), and several services (rabbitmq-exporter, carts, orders, user) show multiple short CPU high runs around ~23:26Z.

The original code execution output of IPython Kernel is also provided below for reference:

(       candidate cmdb_id kpi_name   global_p95  global_p5             timestamp         value  is_high  is_low
0    user|diskio    user   diskio  6428.398179   0.000000  2024-01-20T23:25:00Z  86120.116709     True   False
1    user|diskio    user   diskio  6428.398179   0.000000  2024-01-20T23:26:00Z   8035.497724     True   False
2    user|diskio    user   diskio  6428.398179   0.000000  2024-01-20T23:27:00Z      0.000000    False    True
3    user|diskio    user   diskio  6428.398179   0.000000  2024-01-20T23:28:00Z      0.000000    False    True
4    user|diskio    user   diskio  6428.398179   0.000000  2024-01-20T23:29:00Z      0.000000    False    True
..           ...     ...      ...          ...        ...                   ...           ...      ...     ...
145     user|cpu    user      cpu     0.946160   0.788954  2024-01-20T23:45:00Z      0.885451    False   False
146     user|cpu    user      cpu     0.946160   0.788954  2024-01-20T23:46:00Z      0.835193    False   False
147     user|cpu    user      cpu     0.946160   0.788954  2024-01-20T23:47:00Z      0.897724    False   False
148     user|cpu    user      cpu     0.946160   0.788954  2024-01-20T23:48:00Z      0.921328    False   False
149     user|cpu    user      cpu     0.946160   0.788954  2024-01-20T23:49:00Z      0.947144     True   False

[150 rows x 9 columns],              cmdb_id kpi_name    global_p95     global_p5  points_in_window earliest_high_timestamp  high_run_count
0               user   diskio   6428.398179      0.000000                25    2024-01-20T23:25:00Z               1
1       queue-master   diskio  75947.292512  26343.682367                25    2024-01-20T23:40:00Z               1
2  rabbitmq-exporter      cpu      0.004802      0.000130                25    2024-01-20T23:26:00Z               2
3              carts      cpu      1.782347      1.391448                25    2024-01-20T23:26:00Z               2
4             orders      cpu      1.701799      1.032595                25    2024-01-20T23:26:00Z               2
5               user      cpu      0.946160      0.788954                25    2024-01-20T23:25:00Z               2)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```